# 07: Empirical Validation（经验验证）

目标：用真实微博数据对照验证模型的核心经验假设（H1-H4），并在两套标注数据上做同方法复核：

- **核心对照集**：`outputs/annotations/master/long_covid_annotations_master.jsonl`（17,604）+ `dataset/Topic_data/merged_topic_official.csv`
- **扩展集（Batch3）**：`outputs/annotations/batches/batch_03_expanded/new_batch3.jsonl`（73,456）+ `outputs/annotations/intermediate/to_annotate_batch3_clean.csv`

统一口径：
- 暂不做“按话题分组比较”（后续再补 topic mapping）。
- 先做同一套指标与统计检验，在 **核心对照集 / 扩展集 / 合并样本** 上分别报告结果，检查稳健性。

## H1-H4 与研究问题的对应
- **H1（Activity→Jump）**：$a$ 越高（中立者越少），系统越容易出现突变式变化（用 $|d|Q|/dt|$ 峰值/突变指标衡量）。
- **H2（r_proxy→Volatility）**：$r\_{proxy}$ 越高（自媒体相对更占优），波动性越大（用 $\mathrm{std}(Q)$ / rolling volatility）。
- **H3（r×a 交互）**：高 $r\_{proxy}$ 且高 $a$ 的窗口/分段应更“脆弱”（波动/突变更大）。
- **H4（临界慢化）**：突变前应出现 AC1↑、Var↑ 的早期预警信号（经验数据里可能被外生冲击与噪声掩盖，需客观报告）。


In [ ]:
# ===== 0. Imports & Paths =====
from __future__ import annotations

import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# 设定项目根目录
ROOT = Path("..").resolve() if Path("..").resolve().name == "emotion_dynamics" else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.empirical import load_topic_dataset, aggregate_time_series, UserTypeMapper
from src.empirical.time_series import TimeSeriesConfig, calculate_r_proxy, calculate_rolling_ac1, calculate_rolling_stats

# 输出目录
FIG_DIR = ROOT / "outputs/figs/empirical"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 中文字体（若缺失会自动回退）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("ROOT:", ROOT)
print("FIG_DIR:", FIG_DIR)


In [ ]:
# ===== 1. 读取与对齐：dataset csv + annotations jsonl =====

@dataclass(frozen=True)
class DatasetSpec:
    name: str
    dataset_csv: Path
    annotations_jsonl: Path


def load_annotations_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            records.append(obj)
    if not records:
        raise ValueError(f"标注文件为空：{path}")
    df = pd.DataFrame(records)
    if "mid" not in df.columns:
        raise ValueError(f"标注文件缺少 mid：{path}")
    df["mid"] = df["mid"].astype(str)
    keep = [c for c in ["mid", "emotion_class", "risk_class", "emotion_confidence", "risk_confidence"] if c in df.columns]
    df = df[keep].drop_duplicates(subset=["mid"]).reset_index(drop=True)
    return df


def load_and_merge(spec: DatasetSpec, *, mapper: Optional[UserTypeMapper] = None) -> pd.DataFrame:
    mapper = mapper or UserTypeMapper()
    df_raw = load_topic_dataset(spec.dataset_csv, mapper=mapper)
    if "mid" not in df_raw.columns:
        raise ValueError(f"dataset 缺少 mid 列：{spec.dataset_csv}")
    df_raw["mid"] = df_raw["mid"].astype(str)
    df_raw = df_raw.drop_duplicates(subset=["mid"]).reset_index(drop=True)

    df_ann = load_annotations_jsonl(spec.annotations_jsonl)

    df = df_raw.merge(
        df_ann,
        on=["mid"],
        how="inner",
        validate="one_to_one",
    )
    coverage = len(df) / max(len(df_raw), 1)
    print(f"[{spec.name}] raw={len(df_raw):,} ann={len(df_ann):,} merged={len(df):,} coverage={coverage:.2%}")
    return df


MASTER = DatasetSpec(
    name="master",
    dataset_csv=ROOT / "dataset/Topic_data/merged_topic_official.csv",
    annotations_jsonl=ROOT / "outputs/annotations/master/long_covid_annotations_master.jsonl",
)

BATCH3 = DatasetSpec(
    name="batch3",
    dataset_csv=ROOT / "outputs/annotations/intermediate/to_annotate_batch3_clean.csv",
    annotations_jsonl=ROOT / "outputs/annotations/batches/batch_03_expanded/new_batch3.jsonl",
)

for s in [MASTER, BATCH3]:
    if not s.dataset_csv.exists():
        raise FileNotFoundError(s.dataset_csv)
    if not s.annotations_jsonl.exists():
        raise FileNotFoundError(s.annotations_jsonl)

print("OK: 数据文件存在")


In [ ]:
# ===== 2. 加载两套数据，并构造合并样本 =====

mapper = UserTypeMapper()

df_master = load_and_merge(MASTER, mapper=mapper)
df_batch3 = load_and_merge(BATCH3, mapper=mapper)

df_all = pd.concat([df_master, df_batch3], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["mid"]).reset_index(drop=True)

print("[all] total merged:", f"{len(df_all):,}")
print("time span (master):", df_master["publish_time"].min(), "~", df_master["publish_time"].max())
print("time span (batch3):", df_batch3["publish_time"].min(), "~", df_batch3["publish_time"].max())
print("time span (all):", df_all["publish_time"].min(), "~", df_all["publish_time"].max())

display(df_all[["user_type"]].value_counts().rename("count").head(10))


In [ ]:
# ===== 3. 聚合为时间序列（统一口径） =====

# 方案B（更稳健）：先聚焦数据最密集的时间段（默认 2023+），并用更粗粒度聚合减少缺口伪影
TIME_START = "2023-01-01"  # 可改；建议先聚焦 2023+
TIME_END = None              # 例如 "2024-12-31"；None 表示不截断
FREQ = "4H"                # 推荐：4H（比 1H 更稳健；1D 可作为稳健性对照）
MIN_POSTS_PUBLIC = 5         # 每个窗口至少多少 public 帖子才计算 a/Q（master 较稀疏，阈值过高会导致无法检验；可对 batch3/all 提高到 20 做稳健性对照）

def build_time_series(
    df: pd.DataFrame,
    *,
    freq: str,
    min_posts: int,
    time_start: Optional[str] = None,
    time_end: Optional[str] = None,
) -> pd.DataFrame:
    cfg = TimeSeriesConfig(freq=freq, min_posts=int(min_posts))
    ts = aggregate_time_series(df, config=cfg)
    ts["r_proxy"] = calculate_r_proxy(ts)
    ts = ts.sort_values("time_window").reset_index(drop=True)
    if time_start:
        ts = ts[ts["time_window"] >= pd.Timestamp(time_start)]
    if time_end:
        ts = ts[ts["time_window"] <= pd.Timestamp(time_end)]
    return ts.reset_index(drop=True)

print(f"config: TIME_START={TIME_START}, TIME_END={TIME_END}, FREQ={FREQ}, MIN_POSTS_PUBLIC={MIN_POSTS_PUBLIC}")

ts_master = build_time_series(df_master, freq=FREQ, min_posts=MIN_POSTS_PUBLIC, time_start=TIME_START, time_end=TIME_END)
ts_batch3 = build_time_series(df_batch3, freq=FREQ, min_posts=MIN_POSTS_PUBLIC, time_start=TIME_START, time_end=TIME_END)
ts_all = build_time_series(df_all, freq=FREQ, min_posts=MIN_POSTS_PUBLIC, time_start=TIME_START, time_end=TIME_END)

print("valid windows (master):", int(ts_master["a"].notna().sum()), "/", len(ts_master))
print("valid windows (batch3):", int(ts_batch3["a"].notna().sum()), "/", len(ts_batch3))
print("valid windows (all):", int(ts_all["a"].notna().sum()), "/", len(ts_all))

# 可选：落盘，方便后续复用（不会覆盖旧文件，统一加后缀）
out_dir = ROOT / "outputs/annotations/derived"
out_dir.mkdir(parents=True, exist_ok=True)
ts_master.to_csv(out_dir / f"time_series_master_{FREQ.lower()}.csv", index=False)
ts_batch3.to_csv(out_dir / f"time_series_batch3_{FREQ.lower()}.csv", index=False)
ts_all.to_csv(out_dir / f"time_series_all_{FREQ.lower()}.csv", index=False)
print("saved:", out_dir)


In [ ]:
# ===== 4. 快速可视化：Q / a / r_proxy =====

def plot_basic(ts: pd.DataFrame, title: str, *, out_name: str) -> None:
    df = ts.copy()
    df = df.sort_values("time_window").reset_index(drop=True)
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(df["time_window"], df["Q"], lw=1)
    axes[0].set_ylabel("Q")
    axes[0].set_title(title)
    axes[1].plot(df["time_window"], df["a"], lw=1)
    axes[1].set_ylabel("a")
    axes[2].plot(df["time_window"], df["r_proxy"], lw=1)
    axes[2].set_ylabel("r_proxy")
    axes[2].set_xlabel("time")
    for ax in axes:
        ax.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIG_DIR / out_name, dpi=200)
    plt.show()

plot_basic(ts_master, f"Master: Q/a/r_proxy ({FREQ})", out_name=f"fig7a_master_basic_{FREQ.lower()}.png")
plot_basic(ts_batch3, f"Batch3: Q/a/r_proxy ({FREQ})", out_name=f"fig7a_batch3_basic_{FREQ.lower()}.png")
plot_basic(ts_all, f"All (master+batch3): Q/a/r_proxy ({FREQ})", out_name=f"fig7a_all_basic_{FREQ.lower()}.png")


In [ ]:
# ===== 5. 指标构造：jump / volatility / rolling AC1+Var =====

def _freq_to_step_hours(freq: str) -> float:
    td = pd.to_timedelta(freq)
    return float(td.total_seconds() / 3600.0)

def add_window_metrics(ts: pd.DataFrame, *, freq: str, vol_win: int = 12) -> pd.DataFrame:
    """为窗口级别构造 jump/volatility 指标（方案B：严格处理缺口）。

    - jump 使用：abs_dQ_abs_per_hour = |Δ|Q|| / Δt_hours
    - 仅在“严格连续步长”的窗口对上计算导数；跨缺口/跨 NaN 一律置为 NaN。
    - 构造 block_id：后续 rolling 只在同一连续块内进行，避免压缩时间造成伪影。
    """
    df = ts.copy().sort_values("time_window").reset_index(drop=True)
    step_hours = _freq_to_step_hours(freq)

    df["Q_abs"] = df["Q"].abs()
    df["dt_hours"] = df["time_window"].diff().dt.total_seconds() / 3600.0
    df.loc[df["dt_hours"] <= 0, "dt_hours"] = np.nan

    df["dt_ok"] = np.isclose(df["dt_hours"], step_hours)
    if len(df) > 0:
        df.loc[df.index[0], "dt_ok"] = False
    df.loc[~df["dt_ok"], "dt_hours"] = np.nan

    df["dQ_abs_per_hour"] = df["Q_abs"].diff() / df["dt_hours"]
    df["abs_dQ_abs_per_hour"] = df["dQ_abs_per_hour"].abs()

    valid = df["Q"].notna()
    prev_valid = valid.shift(1, fill_value=False)
    df["is_break"] = (~valid) | (~prev_valid) | (~df["dt_ok"])
    df["block_id"] = df["is_break"].cumsum()

    df["Q_volatility"] = df["Q"].rolling(vol_win, min_periods=max(3, vol_win // 3)).std()
    return df


def segment_metrics(df: pd.DataFrame, *, segment: str = "M", jump_q: float = 0.95) -> pd.DataFrame:
    """把时间序列分段（默认按月），用段内统计量检验 H1-H3（方案B：去极值偏置）。

    - a_mean：默认按 n_public 做加权平均
    - r_proxy_mean：段内媒体计数比值（sum 计数）
    - jump 指标：用段内分位数（默认 q95）替代 max，降低极值偏置
    """
    x = df.dropna(subset=["time_window"]).copy()
    x["seg"] = x["time_window"].dt.to_period(segment).dt.to_timestamp()
    rows = []
    for seg, g in x.groupby("seg"):
        g_aq = g.dropna(subset=["a", "Q"])
        if len(g_aq) < 10:
            continue
        g_jump = g.dropna(subset=["abs_dQ_abs_per_hour"])
        if len(g_jump) < 5:
            continue

        if "n_public" in g_aq.columns:
            w = g_aq["n_public"].fillna(0).astype(float).values
            a_mean = float(np.average(g_aq["a"].values, weights=w)) if float(w.sum()) > 0 else float(g_aq["a"].mean())
        else:
            a_mean = float(g_aq["a"].mean())

        if "n_mainstream" in g_aq.columns and "n_wemedia" in g_aq.columns:
            nw = float(g_aq["n_wemedia"].fillna(0).sum())
            nm = float(g_aq["n_mainstream"].fillna(0).sum())
            r_proxy_mean = float(nw / (nw + nm)) if (nw + nm) > 0 else np.nan
        else:
            r_proxy_mean = float(g_aq["r_proxy"].mean()) if "r_proxy" in g_aq.columns else np.nan

        rows.append(
            {
                "seg": seg,
                "n_windows_aq": int(len(g_aq)),
                "n_windows_jump": int(len(g_jump)),
                "n_public_sum": float(g_aq["n_public"].fillna(0).sum()) if "n_public" in g_aq.columns else np.nan,
                "a_mean": a_mean,
                "r_proxy_mean": r_proxy_mean,
                "volatility": float(g_aq["Q"].std()),
                "jump_q95": float(np.nanpercentile(g_jump["abs_dQ_abs_per_hour"].values, 100.0 * float(jump_q))),
                "jump_max": float(np.nanmax(g_jump["abs_dQ_abs_per_hour"].values)),
            }
        )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values("seg").reset_index(drop=True)


def safe_pearsonr(x: pd.Series, y: pd.Series):
    try:
        from scipy import stats
        m = x.notna() & y.notna()
        if int(m.sum()) < 5:
            return np.nan, np.nan
        r, p = stats.pearsonr(x[m].values, y[m].values)
        return float(r), float(p)
    except Exception:
        return np.nan, np.nan


def safe_spearmanr(x: pd.Series, y: pd.Series):
    try:
        from scipy import stats
        m = x.notna() & y.notna()
        if int(m.sum()) < 5:
            return np.nan, np.nan
        r, p = stats.spearmanr(x[m].values, y[m].values)
        return float(r), float(p)
    except Exception:
        return np.nan, np.nan


def partial_pearsonr(x: pd.Series, y: pd.Series, ctrl: pd.Series):
    m = x.notna() & y.notna() & ctrl.notna()
    if int(m.sum()) < 8:
        return np.nan, np.nan
    xv = x[m].values.astype(float)
    yv = y[m].values.astype(float)
    cv = ctrl[m].values.astype(float)
    X = np.column_stack([np.ones(len(cv)), cv])
    bx, *_ = np.linalg.lstsq(X, xv, rcond=None)
    by, *_ = np.linalg.lstsq(X, yv, rcond=None)
    xr = xv - X @ bx
    yr = yv - X @ by
    return safe_pearsonr(pd.Series(xr), pd.Series(yr))

def run_h1_h2_h3(ts: pd.DataFrame, *, name: str) -> pd.DataFrame:
    df = add_window_metrics(ts, freq=FREQ)
    seg = segment_metrics(df, segment="M", jump_q=0.95)
    if seg.empty:
        print(f"[{name}] 段内有效样本不足，无法检验 H1-H3")
        return seg

    r1, p1 = safe_pearsonr(seg["a_mean"], seg["jump_q95"])  # H1
    rs1, ps1 = safe_spearmanr(seg["a_mean"], seg["jump_q95"])  # H1
    r2, p2 = safe_pearsonr(seg["r_proxy_mean"], seg["volatility"])  # H2
    rs2, ps2 = safe_spearmanr(seg["r_proxy_mean"], seg["volatility"])  # H2

    print(f"\n[{name}] segments={len(seg)}")
    print(f"H1: corr(a_mean, jump_q95) = {r1:.3f} (p={p1:.4f}); spearman={rs1:.3f} (p={ps1:.4f})")
    print(f"H2: corr(r_proxy_mean, volatility) = {r2:.3f} (p={p2:.4f}); spearman={rs2:.3f} (p={ps2:.4f})")

    # 诊断：jump 指标是否仍受样本量影响（极值偏置会导致 jump 与 n_windows 强相关）
    rj, pj = safe_pearsonr(seg["n_windows_jump"], seg["jump_q95"])
    if not np.isnan(rj):
        print(f"diag: corr(n_windows_jump, jump_q95) = {rj:.3f} (p={pj:.4f})")

    # 部分相关：控制段内样本量（最小化 ‘窗口越多→max越大’ 的统计伪影）
    rp1, pp1 = partial_pearsonr(seg["a_mean"], seg["jump_q95"], seg["n_windows_jump"])
    if not np.isnan(rp1):
        print(f"H1(partial, ctrl=n_windows_jump): r={rp1:.3f} (p={pp1:.4f})")

    # H3: 简单分组对照 + 回归（可选 statsmodels）
    a_med = float(seg["a_mean"].median())
    r_med = float(seg["r_proxy_mean"].median())
    hi = seg[(seg["a_mean"] > a_med) & (seg["r_proxy_mean"] > r_med)]
    lo = seg[(seg["a_mean"] <= a_med) & (seg["r_proxy_mean"] <= r_med)]
    print(f"H3(group): high-high n={len(hi)}, mean(vol)={hi['volatility'].mean():.4f}; low-low n={len(lo)}, mean(vol)={lo['volatility'].mean():.4f}")

    try:
        import statsmodels.formula.api as smf
        m = seg.dropna(subset=["volatility", "a_mean", "r_proxy_mean"]).copy()
        if len(m) >= 10:
            model = smf.ols("volatility ~ a_mean * r_proxy_mean", data=m).fit()
            print("H3(reg): volatility ~ a_mean * r_proxy_mean")
            print(model.summary().tables[1])
    except Exception:
        print("H3(reg): statsmodels 不可用，跳过回归（不影响主结论）。")

    # 图：H1/H2 散点
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(seg["a_mean"], seg["jump_q95"], s=25, alpha=0.8)
    axes[0].set_xlabel("a_mean (segment)")
    axes[0].set_ylabel("jump_q95 (q95 |d|Q|/dt|)")
    axes[0].set_title(f"H1 ({name})")
    axes[0].grid(True, alpha=0.2)

    axes[1].scatter(seg["r_proxy_mean"], seg["volatility"], s=25, alpha=0.8)
    axes[1].set_xlabel("r_proxy_mean (segment)")
    axes[1].set_ylabel("volatility (std(Q))")
    axes[1].set_title(f"H2 ({name})")
    axes[1].grid(True, alpha=0.2)

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"fig7b_h1_h2_scatter_{name}_{FREQ.lower()}.png", dpi=200)
    plt.show()

    return seg


seg_master = run_h1_h2_h3(ts_master, name="master")
seg_batch3 = run_h1_h2_h3(ts_batch3, name="batch3")
seg_all = run_h1_h2_h3(ts_all, name="all")


In [ ]:
# ===== 6. H4：临界慢化（事件对齐的 AC1/Var） =====

def rolling_ac1_window(w: np.ndarray) -> float:
    if np.isnan(w).any():
        return np.nan
    w = w - float(np.mean(w))
    x1 = w[:-1]
    x2 = w[1:]
    denom = float(np.sqrt(np.sum(x1**2) * np.sum(x2**2)))
    if denom == 0:
        return 0.0
    return float(np.sum(x1 * x2) / denom)


def add_block_rolling(
    df: pd.DataFrame,
    *,
    col: str,
    window: int,
    block_col: str = "block_id",
) -> pd.DataFrame:
    out = df.copy()
    out[f"{col}_rolling_var"] = np.nan
    out[f"{col}_rolling_ac1"] = np.nan
    for _, g in out.groupby(block_col):
        s = g[col]
        out.loc[g.index, f"{col}_rolling_var"] = s.rolling(int(window), min_periods=int(window)).var()
        out.loc[g.index, f"{col}_rolling_ac1"] = s.rolling(int(window), min_periods=int(window)).apply(rolling_ac1_window, raw=True)
    return out


def pick_jump_events(
    df: pd.DataFrame,
    *,
    q_col: str = "abs_dQ_abs_per_hour",
    quantile: float = 0.95,
    min_gap_windows: int = 6,
    block_col: str = "block_id",
):
    x = df.dropna(subset=["time_window", q_col]).copy().sort_values("time_window")
    if len(x) < 30:
        return []
    thr = float(x[q_col].quantile(float(quantile)))
    cand = x[x[q_col] >= thr]
    step_hours = _freq_to_step_hours(FREQ)
    events = []
    last_time = None
    last_block = None
    for idx, row in cand.iterrows():
        b = row.get(block_col, None)
        if last_time is not None and b == last_block:
            dt_hours = (row["time_window"] - last_time).total_seconds() / 3600.0
            if dt_hours < float(min_gap_windows) * step_hours:
                continue
        events.append(int(idx))
        last_time = row["time_window"]
        last_block = b
    return events


def event_study(
    df: pd.DataFrame,
    event_idx: list[int],
    *,
    col: str,
    pre: int = 24,
    block_col: str = "block_id",
):
    mats = []
    for idx in event_idx:
        if idx - int(pre) < 0:
            continue
        w = df.loc[idx - int(pre) : idx, [col, block_col]]
        if w[block_col].nunique() != 1:
            continue
        arr = w[col].values
        if np.isnan(arr).any():
            continue
        mats.append(arr)
    if not mats:
        return None
    mat = np.vstack(mats)
    mean = np.mean(mat, axis=0)
    lo = np.percentile(mat, 2.5, axis=0)
    hi = np.percentile(mat, 97.5, axis=0)
    return mean, lo, hi, mat


def run_h4(ts: pd.DataFrame, *, name: str, roll_win: int = 12, pre: int = 24):
    df = add_window_metrics(ts, freq=FREQ)

    # 方案B：rolling 只在连续块内计算（避免跨缺口压缩时间造成伪影）；并用 |Q| 提升稳健性
    df = add_block_rolling(df, col="Q_abs", window=int(roll_win), block_col="block_id")

    events = pick_jump_events(
        df,
        q_col="abs_dQ_abs_per_hour",
        quantile=0.95,
        min_gap_windows=max(3, roll_win // 2),
        block_col="block_id",
    )
    step_hours = _freq_to_step_hours(FREQ)
    print(f"[{name}] H4: roll_win={roll_win} (~{roll_win*step_hours:.0f}h), pre={pre} (~{pre*step_hours:.0f}h), events={len(events)}")

    ac_col = "Q_abs_rolling_ac1"
    var_col = "Q_abs_rolling_var"

    ac = event_study(df, events, col=ac_col, pre=pre, block_col="block_id")
    var = event_study(df, events, col=var_col, pre=pre, block_col="block_id")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    xs = np.arange(-pre, 1)

    if ac is not None:
        mean, lo, hi, _ = ac
        axes[0].plot(xs, mean, lw=2)
        axes[0].fill_between(xs, lo, hi, alpha=0.2)
    axes[0].axvline(0, color="gray", linestyle="--", lw=1)
    axes[0].set_title(f"H4: AC1(|Q|) before jumps ({name})")
    axes[0].set_xlabel("windows to jump")
    axes[0].set_ylabel("AC1")
    axes[0].grid(True, alpha=0.2)

    if var is not None:
        mean, lo, hi, _ = var
        axes[1].plot(xs, mean, lw=2)
        axes[1].fill_between(xs, lo, hi, alpha=0.2)
    axes[1].axvline(0, color="gray", linestyle="--", lw=1)
    axes[1].set_title(f"H4: Var(|Q|) before jumps ({name})")
    axes[1].set_xlabel("windows to jump")
    axes[1].set_ylabel("Var")
    axes[1].grid(True, alpha=0.2)

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"fig7c_h4_eventstudy_{name}_{FREQ.lower()}.png", dpi=200)
    plt.show()

run_h4(ts_master, name="master", roll_win=12, pre=24)
run_h4(ts_batch3, name="batch3", roll_win=12, pre=24)
run_h4(ts_all, name="all", roll_win=12, pre=24)


## 结论整理（写作建议）

建议在论文中按“可复现口径”写：
- 先报告三套口径（master / batch3 / all）的 H1-H4 方向是否一致；
- 再解释若 H4 不稳定/不显著：经验数据存在外生冲击、平台机制与观测噪声，可能淹没临界慢化信号（这并不否认理论机制，而是提示识别困难与需要更精细的因果设计）。

下一步（后续补充）：
- 做 topic mapping 后按话题对照；
- 对窗口长度（1H/4H/1D）、min_posts 阈值、jump 定义阈值做稳健性扫描。
